In [1]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
from pathlib import Path

pd.set_option('display.max_columns', None)



In [2]:
# ── 1. PATH CONFIGURATION ────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().resolve()

# Safer project root detection
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

BASE_DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = BASE_DATA_DIR / "raw"
BRONZE_DIR = BASE_DATA_DIR / "bronze"
SILVER_DIR = BASE_DATA_DIR / "silver"

RAW_DIR.mkdir(parents=True, exist_ok=True)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = (
    "https://static.data.gouv.fr/resources/"
    "election-presidentielle-des-10-et-24-avril-2022-resultats-definitifs-du-1er-tour/"
    "20220414-152612/resultats-par-niveau-burvot-t1-france-entiere.xlsx"
)

# Correct raw file path
path_xlsx_raw = RAW_DIR / "2022_burvot_t1_france_entiere.xlsx"
path_rhone_bronze = BRONZE_DIR / "2022-pres-t1-commune-rhone-69-bronze.csv"
path_rhone_silver = SILVER_DIR / "2022-pres-t1-commune-rhone-69-silver.csv"

print(f"Raw data path: {path_xlsx_raw}")
print(f"Bronze path: {path_rhone_bronze}")
print(f"Silver path: {path_rhone_silver}")


Raw data path: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/raw/2022_burvot_t1_france_entiere.xlsx
Bronze path: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/bronze/2022-pres-t1-commune-rhone-69-bronze.csv
Silver path: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/silver/2022-pres-t1-commune-rhone-69-silver.csv


In [3]:
# ── 2. RAW LAYER: Landing Zone ──────────────────────────────────────────────
if not path_xlsx_raw.exists():
    print("📥 Downloading source to RAW...")
    resp = requests.get(DATA_URL, timeout=180)
    resp.raise_for_status()
    with open(path_xlsx_raw, "wb") as f:
        f.write(resp.content)
    print(f"✅ Saved to RAW: {path_xlsx_raw}")
else:
    print(f"✅ Raw file already exists in {RAW_DIR}")


✅ Raw file already exists in /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/raw


In [4]:
# ── 3. BRONZE LAYER: Ingestion & Metadata ───────────────────────────────────
print("\n🛠 Processing RAW -> BRONZE...")
df_all = pd.read_excel(path_xlsx_raw, header=0, dtype=str, engine="openpyxl")

# Identify the department column
col_dept = next(
    (c for c in df_all.columns if any(kw in c.lower() for kw in ["département", "departement", "dept"])),
    None
)

if col_dept is None:
    raise ValueError("Department column not found in source file.")

# Filter for Rhône (69)
df_bronze = df_all[
    df_all[col_dept].astype(str).str.strip().str.lstrip("0") == "69"
].copy()

print("____________________")
print(df_bronze.columns.to_list())

# --- RECONSTRUCT CANDIDATE COLUMNS (Structural Ingestion) ---
CAND_FIELDS = ["N°Panneau", "Sexe", "Nom", "Prénom", "Voix", "% Voix/Ins", "% Voix/Exp"]
cols = list(df_bronze.columns)
start_cand = next((i for i, c in enumerate(cols) if "panneau" in c.lower()), None)

if start_cand is not None:
    new_cols = cols[:start_cand]
    remaining = len(cols) - start_cand
    n_cand = remaining // len(CAND_FIELDS)

    for k in range(1, n_cand + 1):
        for field in CAND_FIELDS:
            new_cols.append(f"{field}_{k}")

    # Keep any leftover columns unchanged
    leftover_start = start_cand + n_cand * len(CAND_FIELDS)
    new_cols.extend(cols[leftover_start:])

    if len(new_cols) != len(cols):
        raise ValueError("Column reconstruction produced a mismatched number of columns.")

    df_bronze.columns = new_cols

# --- ADD BRONZE METADATA ---
df_bronze["extraction_source_url"] = DATA_URL
df_bronze["ingestion_timestamp"] = datetime.now().isoformat()
df_bronze["source_file_name"] = path_xlsx_raw.name





🛠 Processing RAW -> BRONZE...
____________________
['Code du département', 'Libellé du département', 'Code de la circonscription', 'Libellé de la circonscription', 'Code de la commune', 'Libellé de la commune', 'Code du b.vote', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', 'N°Panneau', 'Sexe', 'Nom', 'Prénom', 'Voix', '% Voix/Ins', '% Voix/Exp', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'Unnamed: 61', 'Unnam

In [5]:
# ── 4. SAVE TO BRONZE ───────────────────────────────────────────────────────
df_bronze.to_csv(path_rhone_bronze, index=False, sep=";", encoding="utf-8")

print(df_bronze.head())

print(f"✅ BRONZE dataset created: {path_rhone_bronze}")
print(f"📊 Rows: {len(df_bronze)} | Columns: {len(df_bronze.columns)}")

      Code du département Libellé du département Code de la circonscription  \
47273                  69                  Rhône                         08   
47274                  69                  Rhône                         09   
47275                  69                  Rhône                         05   
47276                  69                  Rhône                         05   
47277                  69                  Rhône                         09   

      Libellé de la circonscription Code de la commune Libellé de la commune  \
47273          8ème circonscription                001                Affoux   
47274          9ème circonscription                002            Aigueperse   
47275          5ème circonscription                003     Albigny-sur-Saône   
47276          5ème circonscription                003     Albigny-sur-Saône   
47277          9ème circonscription                004                  Alix   

      Code du b.vote Inscrits Abstentions % 